# 3.决策树

## 3.1

### 3.1.1信息论


$$l(x_i)=-log_2p(x_i)$$

其中信息$l(x_i)$是指单个类别xi的信息量。 

它描述了一个事件发生时，能给你带来多少 “意外感” 或 “新鲜度”。    

举个例子
- 如果事件是 “太阳从东边升起”（概率$p≈1$），那它发生时，你不会觉得意外，所以信息量I≈0。  
- 如果事件是 “中了彩票头奖”（概率$p≈0$），那它发生时，你会非常意外，所以信息量I会很大。  

$p(x_i)$是概率，$p(x_i)∈[0,1]$所以$log_2p(x_i)<0$，需要加负号。

$$H=-\sum_{i=1}^{n}p(x_i)log_2p(x_i)$$

其中熵$H$是指所有信息量的加权平均值。  

它描述了整个系统里所有事件的 “平均意外感”，或者说系统的 “混乱 / 不确定程度”。  

举个例子
- 抛一枚公平的硬币（正面概率 0.5，反面概率 0.5）：$H=−[0.5log_2​0.5+0.5log_2​0.5]=1$这表示抛硬币的结果有 1 比特的不确定性，很混乱。
- 抛一枚两面都是正面的假硬币（正面概率 1，反面概率 0）：$H=−[1log_2​1+0log_2​0]=0$这表示结果完全确定，没有任何不确定性。





In [1]:
# 数据集
def createDataSet():
    dataSet = [[1, 1, 'yes'],
               [1, 1, 'yes'],
               [1, 0, 'no'],
               [0, 1, 'no'],
               [0, 1, 'no']]
    labels = ['no surfacing', 'flippers']
    return dataSet, labels

## 香农熵的计算公式
香农熵的核心计算步骤：  

- 获取标签：从数据集中提取每个样本的类别标签（即最后一列）。

- 统计数量：计算每个类别出现的次数 labelCounts[key]，以及总样本数 numEntries。

- 计算概率：对每个类别，$p_i$ = 该类次数 / 总样本数。

- 代入公式：$H = -Σ (p_i × log₂(p_i))$，得到最终的熵值。

In [ ]:
# 香农熵计算
from math import log

def calcShannonEnt(dataSet):
    numEntries = len(dataSet)
    labelCounts = {}
    for featVec in dataSet:
        currentLabel = featVec[-1]
        if currentLabel not in labelCounts.keys():
            labelCounts[currentLabel] = 0
        labelCounts[currentLabel] += 1 # labelCounts = {'yes': 2, 'no': 3}
    shannonEnt = 0.0
    for key in labelCounts:
        prob = float(labelCounts[key]) / numEntries # 该类别样本数/总样本数
        shannonEnt -= prob * log(prob, 2) # shannonEnt = - Σ prob * log_2^{prob}
    return shannonEnt

dataSet, labels = createDataSet()
calcShannonEnt(dataSet)

{'yes': 2, 'no': 3}


0.9709505944546686

## 划分数据集
需要划分数据集度量划分数据集的熵，以判断当前是否正确划分数据集  
对每个特征划分数据集的结果计算一次信息熵，然后判断按照哪个特征划分数据是最好的划分方式  

In [ ]:
def splitDataSet(dataSet, axis, value):
    retDataSet = []
    for featVec in dataSet:
        if featVec[axis] == value: # 如果第 axis 个特征等于 指定值
            # 取出该样本中除第 axis 列之外的所有特征
            reducedFeatVec = featVec[:axis]
            reducedFeatVec.extend(featVec[axis+1:])
            retDataSet.append(reducedFeatVec)
    return retDataSet

## 选择最好的数据集划分方式

In [ ]:
def chooseBestFeatureToSplit(dataSet):
    numFeatures = len(dataSet[0]) - 1
    baseEntropy = calcShannonEnt(dataSet)
    bestInfoGain = 0.0
    bestFeature = -1
    for i in range(numFeatures):
        featList = [example[i] for example in dataSet] # example是dataSet中的列表
        uniqueVals = set(featList) # 去重
        newEntropy = 0.0
        for value in uniqueVals:
            subDataSet = splitDataSet(dataSet, i, value) # 得到去除 dataSet 第 i 个特征值为 value 后的子集
            prob = len(subDataSet) / float(len(dataSet)) # 子集权重
            newEntropy += prob * calcShannonEnt(subDataSet)
        infoGain = baseEntropy - newEntropy
        if (infoGain > bestInfoGain):
            bestInfoGain = infoGain
            bestFeature = i
    return bestFeature

## 递归构建决策树

In [4]:
import operator
def majorityCnt(classList):
    classCount = {}
    for vote in classList:
        classCount[vote] = classCount.get(vote, 0) + 1
    sortedClassCount = sorted(classCount.iteritems(), \
                             key = operator.itemgetter(1), reverse = True)
    return sortedClassCount

## 创建树的函数代码

In [ ]:
def createTree(dataSet, labels):
    classList = [example[-1] for example in dataSet] # 提取所有类别标签
    if classList.count(classList[0]) == len(classList):
        return classList[0] # 如果第一个类别出现的次数等于总类别数 则推终止递归
    if len(dataSet[0]) == 1:
        return majorityCnt(classList) # 如果样本行只有1列（标签列） 则通过多数投票 majorityCnt(classList) 返回出现次数最多的类别作为叶子节点
    bestFeat = chooseBestFeatureToSplit(dataSet) # 返回信息量最大的特征索引
    bestFeatLabel = labels[bestFeat] # 返回信息量最大的特征标签
    myTree = {bestFeatLabel:{}} # 构建树 {'no surfacing': {1: {'flippers': {...}}, 0: 'no'}}
    del(labels[bestFeat]) # 删除已使用的特征标签
    featValues = [example[bestFeat] for example in dataSet] # 取所有样本的最佳特征列的值
    uniqueVals = set(featValues) # 去重
    for value in uniqueVals: # 为每个取值递归构建子树
        subLabels = labels[:] # 复制当前剩余的标签列表
        myTree[bestFeatLabel][value] = createTree(splitDataSet\
                                                  (dataSet, bestFeat, value), subLabels) # 
    return myTree